<a href="https://colab.research.google.com/github/upgrade-projects/pysparks-projects/blob/main/Daily_Weather_Observations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Daily Weather Observations**

You are provided with a dataset titled ‘Daily Weather Observations’, which captures atmospheric and meteorological data such as temperature, rainfall, sunshine duration, humidity, wind conditions, pressure and indicators of whether it rained the following day.

With the given [dataset](https://drive.google.com/file/d/1hnwt6t8VstJs0pRGIkR0uF52AxVOcRkJ/view?usp=drive_link), solve the following tasks:

In [11]:
import os

# Install PySpark
!pip install pyspark

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, max, expr

# Initialize SparkSession
spark = SparkSession.builder.appName("WeatherAnalysis").getOrCreate()
print("SparkSession created successfully.")

SparkSession created successfully.


Now, let's mount Google Drive and load the specified dataset into a PySpark DataFrame.

In [13]:
from google.colab import drive
drive.mount('/content/drive')

# The file path provided by the user
file_path = '/content/drive/MyDrive/Colab_Notebooks/Upgrad_Training/data/Weather_Dataset.csv'

# Load data using PySpark, inferring schema and assuming header
df_spark = spark.read.csv(file_path, header=True, inferSchema=True)
df_spark.printSchema()
df_spark.show(5)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
root
 |-- MinTemp: double (nullable = true)
 |-- MaxTemp: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Evaporation: double (nullable = true)
 |-- Sunshine: string (nullable = true)
 |-- WindGustDir: string (nullable = true)
 |-- WindGustSpeed: string (nullable = true)
 |-- WindDir9am: string (nullable = true)
 |-- WindDir3pm: string (nullable = true)
 |-- WindSpeed9am: string (nullable = true)
 |-- WindSpeed3pm: integer (nullable = true)
 |-- Humidity9am: integer (nullable = true)
 |-- Humidity3pm: integer (nullable = true)
 |-- Pressure9am: double (nullable = true)
 |-- Pressure3pm: double (nullable = true)
 |-- Cloud9am: integer (nullable = true)
 |-- Cloud3pm: integer (nullable = true)
 |-- Temp9am: double (nullable = true)
 |-- Temp3pm: double (nullable = true)
 |-- RainToday: string (nullable = true)
 |-- RISK_MM: double (nullabl

Here are the PySpark implementations for each of your tasks:

**Task 1: Write a function that returns the number of days when it rained the next day. Hint: Count the number of rows where RainTomorrow is ‘Yes’.**

In [14]:
def count_rain_tomorrow_yes_pyspark(dataframe):
    """
    Counts the number of days when it rained the next day using PySpark.
    """
    if 'RainTomorrow' in dataframe.columns:
        return dataframe.filter(col('RainTomorrow') == 'Yes').count()
    else:
        return "'RainTomorrow' column not found."

# Task 1 Demonstration
rain_tomorrow_count = count_rain_tomorrow_yes_pyspark(df_spark)
print(f"Number of days when it rained the next day (PySpark): {rain_tomorrow_count}")

Number of days when it rained the next day (PySpark): 66


**Task 2: Write a function that returns the average sunshine duration on days with no rainfall. Hint: Filter rows where Rainfall is 0 and then calculate the average of Sunshine.**

In [15]:
def average_sunshine_no_rainfall_pyspark(dataframe):
    """
    Calculates the average sunshine duration on days with no rainfall using PySpark.
    """
    if 'Rainfall' in dataframe.columns and 'Sunshine' in dataframe.columns:
        # Cast 'Sunshine' column to Double, treating unparseable values (like 'NA') as null
        df_processed = dataframe.withColumn('Sunshine_Double', expr("try_cast(Sunshine as double)"))

        # Filter for days with no rainfall and calculate the average of 'Sunshine_Double'
        no_rainfall_days_avg_sunshine = df_processed.filter(col('Rainfall') == 0)
        avg_sunshine_result = no_rainfall_days_avg_sunshine.agg(avg('Sunshine_Double')).collect()[0][0]
        return avg_sunshine_result
    else:
        return "'Rainfall' or 'Sunshine' column not found."

# Task 2 Demonstration
avg_sunshine = average_sunshine_no_rainfall_pyspark(df_spark)
print(f"Average sunshine duration on days with no rainfall (PySpark): {avg_sunshine:.2f} hours")

Average sunshine duration on days with no rainfall (PySpark): 8.47 hours


**Task 3: Write a function that returns the maximum temperature recorded at 3 PM. Hint: Use the Temp3pm column and return the max value.**

In [16]:
def max_temp_3pm_pyspark(dataframe):
    """
    Returns the maximum temperature recorded at 3 PM using PySpark.
    """
    if 'Temp3pm' in dataframe.columns:
        max_temp_result = dataframe.agg(max('Temp3pm')).collect()[0][0]
        if max_temp_result is not None:
            return max_temp_result
        else:
            return "No valid maximum temperature found in 'Temp3pm' column."
    else:
        return "'Temp3pm' column not found."

# Task 3 Demonstration
max_temp = max_temp_3pm_pyspark(df_spark)
print(f"Maximum temperature recorded at 3 PM (PySpark): {max_temp}°C")

Maximum temperature recorded at 3 PM (PySpark): 34.5°C


**Task 4: Write a function that returns the average humidity at 3 PM on days it rained the next day. Hint: Filter rows where RainTomorrow is ‘Yes’ and then calculate the mean of Humidity3pm.**

In [17]:
def average_humidity_3pm_rain_tomorrow_pyspark(dataframe):
    """
    Returns the average humidity at 3 PM on days it rained the next day using PySpark.
    """
    if 'RainTomorrow' in dataframe.columns and 'Humidity3pm' in dataframe.columns:
        # Filter for days where it rained tomorrow and calculate the mean of 'Humidity3pm'
        rain_tomorrow_days = dataframe.filter(col('RainTomorrow') == 'Yes')
        avg_humidity_result = rain_tomorrow_days.agg(avg('Humidity3pm')).collect()[0][0]
        if avg_humidity_result is not None:
            return avg_humidity_result
        else:
            return "No valid average humidity found for days it rained the next day."
    else:
        return "'RainTomorrow' or 'Humidity3pm' column not found."

# Task 4 Demonstration
avg_humidity = average_humidity_3pm_rain_tomorrow_pyspark(df_spark)
print(f"Average humidity at 3 PM on days it rained the next day (PySpark): {avg_humidity:.2f}%")

Average humidity at 3 PM on days it rained the next day (PySpark): 57.68%


**Task 5: Write a function that returns the most common wind direction at 9 AM on cloudy days. Hint: Filter rows where Cloud9am is above a threshold (e.g. >5) and then find the mode of WindDir9am.**

In [18]:
from pyspark.sql.functions import mode, count, col

def most_common_wind_direction_cloudy_9am_pyspark(dataframe, cloud_threshold=5):
    """
    Returns the most common wind direction at 9 AM on cloudy days using PySpark.
    Cloudy days are defined as Cloud9am values above a specified threshold.
    """
    if 'Cloud9am' in dataframe.columns and 'WindDir9am' in dataframe.columns:
        # Filter for cloudy days (Cloud9am > threshold) and drop rows where WindDir9am is null
        # Also dropna on 'Cloud9am' itself to be safe if it contains nulls before filtering
        cloudy_days = dataframe.filter(col('Cloud9am') > cloud_threshold).dropna(subset=['Cloud9am', 'WindDir9am'])
        print(f"DEBUG: Number of cloudy days found: {cloudy_days.count()}")

        # Group by WindDir9am, count occurrences, order by count descending, and take the first (most common)
        most_common_direction_df = cloudy_days.groupBy('WindDir9am').agg(count('WindDir9am').alias('count')).orderBy(col('count').desc())

        if most_common_direction_df.count() > 0:
            result = most_common_direction_df.select('WindDir9am').limit(1).collect()[0][0]
            print(f"DEBUG: Most common wind direction result: {result}")
            return result
        else:
            print("DEBUG: No common wind direction found after grouping.")
            return "No common wind direction found for cloudy days or insufficient data."
    else:
        return "'Cloud9am' or 'WindDir9am' column not found."

# Task 5 Demonstration
most_common_wind = most_common_wind_direction_cloudy_9am_pyspark(df_spark)
print(f"Most common wind direction at 9 AM on cloudy days (PySpark): {most_common_wind}")

DEBUG: Number of cloudy days found: 151
DEBUG: Most common wind direction result: SSE
Most common wind direction at 9 AM on cloudy days (PySpark): SSE
